# MJD Transformer Results

Loads completed run artifacts only. This notebook never prepares data or trains/evaluates a model.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

configured_root = os.environ.get('MJD_TRANSFORMER_PROJECT_ROOT')
candidate_roots = []
if configured_root:
    candidate_roots.append(Path(configured_root).expanduser())
candidate_roots.extend([
    Path.cwd() / 'mjd_detector',
    Path.cwd(),
    Path.cwd().parent / 'mjd_detector',
    Path.cwd().parent,
    Path.cwd().parent.parent / 'mjd_detector',
])
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in candidate_roots
     if (candidate / 'mjd_transformer').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate mjd_detector; set MJD_TRANSFORMER_PROJECT_ROOT.'
    )
OUTPUT_ROOT = Path(os.environ.get(
    'MJD_OUTPUT_ROOT',
    str(PROJECT_ROOT / 'results' / 'transformer_official_v1'),
)).expanduser()
SUMMARY_PATH = OUTPUT_ROOT / 'transformer_results.csv'
print('Results:', OUTPUT_ROOT)

In [ ]:
def load_completed_runs(output_root):
    rows = []
    for run_dir in sorted(output_root.iterdir() if output_root.is_dir() else []):
        summary_path = run_dir / 'run_summary.json'
        if summary_path.is_file():
            rows.append(json.loads(summary_path.read_text(encoding='utf-8')))
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(
        ['task', 'tokenization', 'position_encoding']
    ).reset_index(drop=True)

results = load_completed_runs(OUTPUT_ROOT)
if results.empty and SUMMARY_PATH.is_file():
    results = pd.read_csv(SUMMARY_PATH)

expected_runs = 12
print(f'Completed runs: {len(results)}/{expected_runs}')
display(results)

In [ ]:
if results.empty:
    print('No completed runs yet.')
else:
    classification_columns = [
        'tokenization', 'position_encoding', 'test_auc', 'test_accuracy',
        'best_epoch', 'epochs_completed', 'minutes_per_epoch',
        'parameter_count', 'test_events',
    ]
    regression_columns = [
        'tokenization', 'position_encoding', 'test_rmse_kev', 'test_mae_kev',
        'test_bias_kev', 'best_epoch', 'epochs_completed',
        'minutes_per_epoch', 'parameter_count', 'test_events',
    ]
    classification = results.loc[results['task'] == 'classification'].copy()
    regression = results.loc[results['task'] == 'regression'].copy()
    if not classification.empty:
        classification = classification.sort_values('test_auc', ascending=False)
        print('Classification — higher AUC is better')
        display(classification[classification_columns])
    if not regression.empty:
        regression = regression.sort_values('test_rmse_kev', ascending=True)
        print('Regression — lower RMSE is better')
        display(regression[regression_columns])

In [ ]:
if not results.empty:
    plot_data = results.copy()
    plot_data['representation'] = (
        plot_data['tokenization'] + '\n' + plot_data['position_encoding']
    )
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    classification_plot = plot_data.loc[plot_data['task'] == 'classification']
    regression_plot = plot_data.loc[plot_data['task'] == 'regression']
    if not classification_plot.empty:
        axes[0].bar(classification_plot['representation'], classification_plot['test_auc'])
        axes[0].set_title('MJD clean/non-clean classification')
        axes[0].set_ylabel('Test ROC-AUC')
        lower = max(0.0, float(classification_plot['test_auc'].min()) - 0.02)
        axes[0].set_ylim(lower, 1.0)
        axes[0].tick_params(axis='x', rotation=45)
    else:
        axes[0].set_visible(False)
    if not regression_plot.empty:
        axes[1].bar(regression_plot['representation'], regression_plot['test_rmse_kev'])
        axes[1].set_title('MJD clean-event energy regression')
        axes[1].set_ylabel('Test RMSE (keV)')
        axes[1].tick_params(axis='x', rotation=45)
    else:
        axes[1].set_visible(False)
    fig.tight_layout()
    plt.show()

In [ ]:
history_rows = []
for run_dir in sorted(OUTPUT_ROOT.iterdir() if OUTPUT_ROOT.is_dir() else []):
    history_path = run_dir / 'history.json'
    config_path = run_dir / 'run_config.json'
    if not history_path.is_file() or not config_path.is_file():
        continue
    history = json.loads(history_path.read_text(encoding='utf-8'))
    config = json.loads(config_path.read_text(encoding='utf-8'))
    for epoch in history:
        metrics = epoch['validation_metrics']
        history_rows.append({
            'run_id': config['run_id'],
            'task': config['task'],
            'epoch': epoch['epoch'],
            'validation_loss': epoch['validation_loss'],
            'validation_auc': metrics.get('macro_auc'),
            'validation_rmse_kev': metrics.get('rmse_kev'),
        })
history_table = pd.DataFrame(history_rows)

if not history_table.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    classification_history = history_table.loc[history_table['task'] == 'classification']
    regression_history = history_table.loc[history_table['task'] == 'regression']
    for run_id, group in classification_history.groupby('run_id'):
        axes[0].plot(group['epoch'], group['validation_auc'], label=run_id)
    axes[0].set_title('Classification validation AUC')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('ROC-AUC')
    if not classification_history.empty:
        axes[0].legend(fontsize=7)
    for run_id, group in regression_history.groupby('run_id'):
        axes[1].plot(group['epoch'], group['validation_rmse_kev'], label=run_id)
    axes[1].set_title('Regression validation RMSE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('RMSE (keV)')
    if not regression_history.empty:
        axes[1].legend(fontsize=7)
    fig.tight_layout()
    plt.show()